# <h1 style="text-align: center; font-weight: bolder;">LAB 8</h1>
# <h1 style="text-align: center; font-weight: bold;">TRANSFORMERS</h1>

This notebook trains a transformer encoder model on MagnaTagATune (MTAT) for multi-label music tagging.
It is self-contained: installs dependencies, loads MTAT (local if available), trains, evaluates with ROC-AUC and PR-AUC,
and reports compute/resource + emissions metrics.

URL for Google Colab: 

URL for GitHub repo: 

# <h2 style="text-align: center; font-weight: bold;">SETUP</h2>


# <h3 style="text-align: left; font-weight: bold;">DEPENDENCIES</h3>


In [1]:
import sys, platform, os, time, math, random

In [2]:
def in_colab():
    return "COLAB_GPU" in os.environ or "google.colab" in sys.modules

def pip_install(pkgs):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

# Minimal deps
pip_install([
    "torch", "torchaudio",
    "datasets[audio]==3",
    "tqdm", "scikit-learn", "matplotlib",
    "psutil",
    "codecarbon"
])

In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from torch.utils.data import Dataset, DataLoader
from torchaudio.transforms import MelSpectrogram, Resample
from tqdm.auto import tqdm, trange

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import roc_auc_score, average_precision_score

import psutil
from codecarbon import OfflineEmissionsTracker

In [4]:
class color:
   PURPLE = '\033[95m'
   CYAN = '\033[96m'
   DARKCYAN = '\033[36m'
   BLUE = '\033[94m'
   GREEN = '\033[92m'
   YELLOW = '\033[93m'
   RED = '\033[91m'
   BOLD = '\033[1m'
   UNDERLINE = '\033[4m'
   END = '\033[0m'

# <h3 style="text-align: left; font-weight: bold;">DEVICES (M1, CUDA OR CPU) & SCALABILITY</h3>


In [5]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    # Mac M1
    device = "mps"   
else:
    device = "cpu"
device

'mps'

# <h3 style="text-align: left; font-weight: bold;">CONFIG</h3>


In [6]:
CFG = {
    # audio
    "sample_rate": 16000,
    # más corto = más rápido?
    "clip_seconds": 2.0,     
    "n_fft": 1024,
    "n_mels": 64,

    # training speed knobs
    "batch_size": 32,
    "epochs": 3,
    "lr": 1e-3,
    # M1 suele ir mejor con 0
    "num_workers": 0 if device in ["mps", "cpu"] else 2,  
    "pin_memory": True if device == "cuda" else False,

    # dataset speed knobs
    "max_train_items": 8000,  
    "max_val_items":  2000,
    "max_test_items": 2000,

    # model
    "n_blocks": 2,
    "emb_dim": 64,
    "n_heads": 4,
    "stride": (4,4),
    "pos_enc_gain": 0.5,
    "cls_token_ratio": 0.25,

    # early stop
    "early_stop_patience": 3
}
CFG

{'sample_rate': 16000,
 'clip_seconds': 2.0,
 'n_fft': 1024,
 'n_mels': 64,
 'batch_size': 32,
 'epochs': 3,
 'lr': 0.001,
 'num_workers': 0,
 'pin_memory': False,
 'max_train_items': 8000,
 'max_val_items': 2000,
 'max_test_items': 2000,
 'n_blocks': 2,
 'emb_dim': 64,
 'n_heads': 4,
 'stride': (4, 4),
 'pos_enc_gain': 0.5,
 'cls_token_ratio': 0.25,
 'early_stop_patience': 3}

# <h3 style="text-align: left; font-weight: bold;">LOAD MTAT</h3>


It'll take local route as prefered way.

# <h4 style="text-align: left; font-weight: bold;">Route resolver</h4>


In [7]:
from pathlib import Path

LOCAL_CANDIDATES = [
    Path("data") / "mtat",
    Path("mtat"),
]

local_root = None
for p in LOCAL_CANDIDATES:
    if p.exists():
        local_root = p.resolve()
        break

local_root

# <h4 style="text-align: left; font-weight: bold;">MTAT Loader</h4>


In [8]:
import os, random
from pathlib import Path
import pandas as pd
import numpy as np

def read_csv_flexible(path):
    try:
        return pd.read_csv(path, sep="\t")
    except Exception:
        return pd.read_csv(path, sep=",")

def make_splits_from_ids(ids, split=(0.8, 0.1, 0.1), seed=SEED):
    ids = list(ids)
    rng = random.Random(seed)
    rng.shuffle(ids)
    n = len(ids)
    n_train = int(split[0] * n)
    n_val = int(split[1] * n)
    train_ids = ids[:n_train]
    val_ids = ids[n_train:n_train + n_val]
    test_ids = ids[n_train + n_val:]
    return train_ids, val_ids, test_ids

def find_datalabs_root():
    anchors = [Path.cwd()]
    env_root = os.environ.get("MTAT_ROOT")
    if env_root:
        anchors.append(Path(env_root))

    for a in anchors:
        for p in [a] + list(a.parents):
            for cand in [
                p / "data" / "datalabs",
                p / "datalabs",
                p / "data" / "mtat",
                p / "mtat",
            ]:
                if (cand / "annotations_final.csv").exists() and (cand / "clip_info_final.csv").exists():
                    return cand
    raise FileNotFoundError("annotations_final.csv not found")

def resolve_audio_root(base):
    candidates = [
        base / "audio",
        base.parent / "audio",
        base / "mp3",
        base.parent / "mp3",
        base / "audios",
        base.parent / "audios",
    ]
    for c in candidates:
        if c.exists() and c.is_dir():
            return c
    raise FileNotFoundError("audio folder not found near datalabs/mtat root")

def normalize_mp3_rel(mp3_path):
    mp3_rel = str(mp3_path).strip()
    if mp3_rel.startswith("audio/"):
        mp3_rel = mp3_rel[len("audio/"):]
    if mp3_rel.startswith("./"):
        mp3_rel = mp3_rel[2:]
    return mp3_rel

def load_mtat():
    base = find_datalabs_root()
    ann_path = base / "annotations_final.csv"
    info_path = base / "clip_info_final.csv"
    audio_root = resolve_audio_root(base)

    df_ann = read_csv_flexible(ann_path)
    df_info = read_csv_flexible(info_path)

    df_ann.columns = [c.strip() for c in df_ann.columns]
    df_info.columns = [c.strip() for c in df_info.columns]

    tag_cols = [c for c in df_ann.columns if c != "clip_id" and df_ann[c].dtype != object]
    ann_idx = df_ann.set_index("clip_id")

    info_idx = df_info.set_index("clip_id")

    all_ids = df_ann["clip_id"].values
    train_ids, val_ids, test_ids = make_splits_from_ids(all_ids)

    def build_df(id_list):
        rows = []
        for cid in id_list:
            if cid in info_idx.index:
                mp3_rel = normalize_mp3_rel(info_idx.loc[cid, "mp3_path"])
                rows.append({"clip_id": cid, "mp3_path": mp3_rel})
        return pd.DataFrame(rows)

    train_df = build_df(train_ids).reset_index(drop=True)
    val_df = build_df(val_ids).reset_index(drop=True)
    test_df = build_df(test_ids).reset_index(drop=True)

    def filter_existing(df):
        paths = (df["mp3_path"].astype(str)).tolist()
        exists = [(audio_root / p).is_file() for p in paths]
        return df.loc[exists].reset_index(drop=True)

    train_df = filter_existing(train_df)
    val_df = filter_existing(val_df)
    test_df = filter_existing(test_df)

    return ann_idx, tag_cols, audio_root, train_df, val_df, test_df

ann_idx, TAG_COLS, AUDIO_ROOT, train_df, val_df, test_df = load_mtat()

# <h3 style="text-align: left; font-weight: bold;">SUBSAMPLE</h3>


In [9]:
def subsample_df(df, max_items):
    if (max_items is None) or (len(df) <= max_items):
        return df
    idx = np.random.RandomState(SEED).choice(len(df), size=max_items, replace=False)
    return df.iloc[sorted(idx)].reset_index(drop=True)

train_df = subsample_df(train_df, CFG["max_train_items"])
val_df = subsample_df(val_df, CFG["max_val_items"])
test_df = subsample_df(test_df, CFG["max_test_items"])

len(train_df), len(val_df), len(test_df)

(8000, 2000, 2000)

# <h3 style="text-align: left; font-weight: bold;">LABELING</h3>


In [10]:
n_classes = len(TAG_COLS)
n_classes, TAG_COLS[:10]

(188,
 ['no voice',
  'singer',
  'duet',
  'plucking',
  'hard rock',
  'world',
  'bongos',
  'harpsichord',
  'female singing',
  'clasical'])

# <h3 style="text-align: left; font-weight: bold;">DATASET AND FEATURE EXTRACTOR</h3>


**Dataset**

In [11]:
import soundfile as sf

class MTATDataset(Dataset):
    """
    finally Robust MTAT dataset for my environment:
    * Filters out missing/bad audio paths at init time.
    * If a read still fails (broken symlink, corrupted file, race, etc) retries with another item.
    """
    def __init__(
        self, df, ann_idx, tag_cols, audio_root,
        sample_rate=16000, clip_seconds=2.0,
        device="cpu", split="train",
        max_retries=8
    ):
        self.audio_root = Path(audio_root)
        self.ann_idx = ann_idx
        self.tag_cols = list(tag_cols)
        self.sample_rate = int(sample_rate)
        self.n_samples = int(sample_rate * clip_seconds)
        self.device = device
        self.split = split
        self.max_retries = int(max_retries)

        # Make a clean copy
        df = df.reset_index(drop=True).copy()
        df["mp3_path"] = df["mp3_path"].astype(str).str.strip()

        # Filter rows whose audio file exists & is a regular file
        ok = []
        for p in df["mp3_path"].tolist():
            fp = self.audio_root / p
            ok.append(fp.is_file())
        self.df = df.loc[ok].reset_index(drop=True)

        if len(self.df) == 0:
            raise RuntimeError(
                f"MTATDataset has 0 valid audio files. "
                f"audio_root={self.audio_root} (check folder + mp3_path format)"
            )

    def __len__(self):
        return len(self.df)

    def _pad_or_crop(self, x):
        if x.numel() < self.n_samples:
            x = F.pad(x, (0, self.n_samples - x.numel()))
        elif x.numel() > self.n_samples:
            if self.split == "train":
                start = random.randint(0, x.numel() - self.n_samples)
            else:
                start = (x.numel() - self.n_samples) // 2
            x = x[start:start + self.n_samples]
        return x

    def _load_audio(self, path: Path):
        audio_np, sr = sf.read(str(path), always_2d=True)
        x = torch.tensor(audio_np, dtype=torch.float32).T.mean(dim=0)  # mono
        if sr != self.sample_rate:
            x = torchaudio.functional.resample(x.unsqueeze(0), sr, self.sample_rate).squeeze(0)
        return x

    def __getitem__(self, idx):
        # retry loop to avoid training crash on occasional bad files: mtat has a looooot
        for _ in range(self.max_retries):
            row = self.df.iloc[int(idx)]
            clip_id = row["clip_id"]
            mp3_rel = str(row["mp3_path"]).strip()
            path = self.audio_root / mp3_rel

            try:
                x = self._load_audio(path)
                x = self._pad_or_crop(x)

                y = self.ann_idx.loc[clip_id, self.tag_cols].values.astype(np.float32)
                y = torch.tensor(y, dtype=torch.float32)

                return {"audio": x.to(self.device), "labels": y.to(self.device)}

            except Exception:
                # pick another sample
                idx = random.randrange(len(self.df))

        row = self.df.iloc[0]
        clip_id = row["clip_id"]
        y = self.ann_idx.loc[clip_id, self.tag_cols].values.astype(np.float32)
        y = torch.tensor(y, dtype=torch.float32)
        x = torch.zeros(self.n_samples, dtype=torch.float32)
        return {"audio": x.to(self.device), "labels": y.to(self.device)}

train_dataset = MTATDataset(train_df, ann_idx, TAG_COLS, AUDIO_ROOT, CFG["sample_rate"], CFG["clip_seconds"], device=device, split="train")
val_dataset   = MTATDataset(val_df,   ann_idx, TAG_COLS, AUDIO_ROOT, CFG["sample_rate"], CFG["clip_seconds"], device=device, split="val")
test_dataset  = MTATDataset(test_df,  ann_idx, TAG_COLS, AUDIO_ROOT, CFG["sample_rate"], CFG["clip_seconds"], device=device, split="test")

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=CFG["num_workers"],
    pin_memory=CFG["pin_memory"],
    drop_last=True
)
val_loader   = DataLoader(
    val_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=CFG["pin_memory"]
)
test_loader  = DataLoader(
    test_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=CFG["pin_memory"]
)

print("Valid items:", len(train_dataset), len(val_dataset), len(test_dataset))

Valid items: 8000 2000 2000


**Feature extractor**

In [12]:
class FeatureExtractor(nn.Module):
    def __init__(self, n_mels=64, n_fft=1024):
        super().__init__()
        self.melspec = MelSpectrogram(sample_rate=CFG["sample_rate"], n_mels=n_mels, n_fft=n_fft)

    def forward(self, x):
        x = self.melspec(x)
        return torch.log10(1 + 1000 * x)

feature_extractor = FeatureExtractor(CFG["n_mels"], CFG["n_fft"]).to(device)

# <h3 style="text-align: left; font-weight: bold;">TRANSFORMER MODEL</h3>


In [13]:
class Transformer(nn.Module):
    def __init__(self, kernel=(8,8), stride=(4,4), n_blocks=2, emb_dim=64, n_heads=4, mlp_ratio=2,
                 output_size=50, pos_enc_gain=0.5, cls_token_ratio=0.25):
        super().__init__()
        self.emb_dim = emb_dim
        self.output_size = output_size
        self.pos_enc_gain = pos_enc_gain
        self.cls_token_ratio = cls_token_ratio

        self.norm = nn.BatchNorm2d(1)
        padding = kernel[0] // 2
        self.patching = nn.Conv2d(1, emb_dim, kernel_size=kernel, stride=stride, padding=padding)

        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                emb_dim, n_heads,
                dim_feedforward=emb_dim * mlp_ratio,
                dropout=0.1,
                norm_first=True
            ) for _ in range(n_blocks)
        ])

        self.cls = nn.Parameter(torch.randn(1,1,emb_dim))
        nn.init.normal_(self.cls, std=0.02)

        self.fc1 = nn.Linear(emb_dim, emb_dim)
        self.fc2 = nn.Linear(emb_dim, output_size)

    def positional_encoding(self, seq_len):
        position = torch.arange(seq_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, self.emb_dim, 2, dtype=torch.float32) * -(math.log(10000.0) / self.emb_dim))
        pe = torch.zeros(seq_len, self.emb_dim, device=self.cls.device)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.norm(x)
        x = self.patching(x)  
        B, E, M, T = x.shape

        pe_m = self.positional_encoding(M)
        pe_t = self.positional_encoding(T)
        pe_m = pe_m.transpose(0,1).unsqueeze(0).unsqueeze(3)
        pe_t = pe_t.transpose(0,1).unsqueeze(0).unsqueeze(2)
        x = x + self.pos_enc_gain * (pe_m + pe_t)

        x = x.flatten(start_dim=2).swapaxes(1,2)  
        cls = self.cls.repeat(B,1,1)
        x = torch.cat([cls, x], dim=1)

        for blk in self.blocks:
            x = blk(x)

        x_cls = x[:,0,:]
        x_mean = x[:,1:,:].mean(dim=1)
        x = self.cls_token_ratio * x_cls + (1 - self.cls_token_ratio) * x_mean

        x = F.layer_norm(x, (self.emb_dim,))
        x = F.relu(self.fc1(x))
        logits = self.fc2(x)  
        return logits

In [14]:
model = Transformer(
    n_blocks=CFG["n_blocks"],
    emb_dim=CFG["emb_dim"],
    n_heads=CFG["n_heads"],
    stride=CFG["stride"],
    pos_enc_gain=CFG["pos_enc_gain"],
    cls_token_ratio=CFG["cls_token_ratio"],
    output_size=n_classes
).to(device)

sum(p.numel() for p in model.parameters() if p.requires_grad)

87550

# <h2 style="text-align: center; font-weight: bold;">RESOURCE MONITOR AND ENVIRONMENTAL IMPACT</h2>

**Resource monitor (CPU, RAM and time)**

In [ ]:
class ResourceMonitor:
    """
    aiming to track CPU%, RAM and the optional accelerator memory if available.
    so in summary:
    * CPU%: psutil.cpu_percent()
    * RAM MB: process RSS
    * GPU memory (CUDA): torch.cuda.memory_allocated()
    * MPS memory (Apple M1): torch.mps.current_allocated_memory() if available
    """
    def __init__(self, interval_s=1.0):
        self.interval_s = interval_s
        self.proc = psutil.Process(os.getpid())
        self.running = False
        self.samples = []
        self._t0 = None
        self._thread = None

    def _get_accel_mem_mb(self):
        # CUDA
        if torch.cuda.is_available():
            try:
                return float(torch.cuda.memory_allocated() / (1024**2))
            except Exception:
                return None
        # MPS (Apple Silicon or M1 technically)
        if hasattr(torch, "mps") and torch.backends.mps.is_available():
            try:
                if hasattr(torch.mps, "current_allocated_memory"):
                    return float(torch.mps.current_allocated_memory() / (1024**2))
            except Exception:
                return None
        return None

    def start(self):
        import threading
        self.running = True
        self._t0 = time.time()

        def loop():
            while self.running:
                t = time.time() - self._t0
                cpu = psutil.cpu_percent(interval=None)
                ram_mb = self.proc.memory_info().rss / (1024**2)
                accel_mb = self._get_accel_mem_mb()
                self.samples.append((t, cpu, ram_mb, accel_mb))
                time.sleep(self.interval_s)

        self._thread = threading.Thread(target=loop, daemon=True)
        self._thread.start()

    def stop(self):
        self.running = False
        time.sleep(self.interval_s)

    def summary(self):
        if not self.samples:
            return {}
        arr = np.array([[s[0], s[1], s[2], (s[3] if s[3] is not None else np.nan)] for s in self.samples], dtype=float)
        out = {
            "wall_time_s": float(arr[-1, 0]),
            "cpu_mean_pct": float(np.nanmean(arr[:, 1])),
            "cpu_max_pct": float(np.nanmax(arr[:, 1])),
            "ram_mean_mb": float(np.nanmean(arr[:, 2])),
            "ram_max_mb": float(np.nanmax(arr[:, 2])),
            "n_samples": int(len(self.samples)),
        }
        if not np.all(np.isnan(arr[:, 3])):
            out.update({
                "accel_mem_mean_mb": float(np.nanmean(arr[:, 3])),
                "accel_mem_max_mb": float(np.nanmax(arr[:, 3])),
            })
        return out

# <h3 style="text-align: left; font-weight: bold;">ROC AUC AND PR AUC METRICS </h3>

ROC-AUD and PR-AUC metrics plus fast training with early stop.

In [16]:
def forward_batch(batch):
    x = batch["audio"].to(device)
    y = batch["labels"].to(device)
    x = feature_extractor(x)
    logits = model(x)
    probs = torch.sigmoid(logits)
    return logits, probs, y

def eval_auc(loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for batch in loader:
            _, probs, y = forward_batch(batch)
            ys.append(y.detach().cpu().numpy())
            ps.append(probs.detach().cpu().numpy())

    y_true = np.concatenate(ys, axis=0)
    y_score = np.concatenate(ps, axis=0)

    pos = y_true.sum(axis=0)
    n = y_true.shape[0]
    valid = (pos > 0) & (pos < n)

    n_valid = int(valid.sum())
    n_total = int(y_true.shape[1])

    if n_valid == 0:
        return float("nan"), float("nan"), {"valid_classes": 0, "total_classes": n_total, "dropped_classes": n_total}

    y_true_v = y_true[:, valid]
    y_score_v = y_score[:, valid]

    roc = roc_auc_score(y_true_v, y_score_v, average="macro")
    pr  = average_precision_score(y_true_v, y_score_v, average="macro")

    diag = {
        "valid_classes": n_valid,
        "total_classes": n_total,
        "dropped_classes": n_total - n_valid
    }
    return float(roc), float(pr), diag

os.makedirs("emissions", exist_ok=True)
tracker = OfflineEmissionsTracker(
    country_iso_code="ESP",
    output_dir="emissions",
    output_file="codecarbon_emissions.csv",
    log_level="error"
)
monitor = ResourceMonitor(interval_s=1.0)

def train_loop():
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
    loss_fn = nn.BCEWithLogitsLoss()

    best_pr = -1.0
    bad_epochs = 0
    history = []
    best_state = None

    tracker.start()
    monitor.start()

    try:
        for epoch in range(CFG["epochs"]):
            t0 = time.time()
            model.train()
            total_loss = 0.0

            pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CFG['epochs']}", leave=True)
            for step, batch in enumerate(pbar, start=1):
                logits, _, y = forward_batch(batch)
                loss = loss_fn(logits, y)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item()
                pbar.set_postfix(loss=f"{loss.item():.4f}", avg=f"{(total_loss/step):.4f}")

            train_loss = total_loss / max(1, len(train_loader))
            val_roc, val_pr, val_diag = eval_auc(val_loader)
            dt = time.time() - t0

            history.append({
                "epoch": epoch+1,
                "train_loss": float(train_loss),
                "val_roc": float(val_roc),
                "val_pr": float(val_pr),
                "epoch_time_s": float(dt),
                "val_valid_tags": int(val_diag["valid_classes"]),
                "val_total_tags": int(val_diag["total_classes"]),
                "val_dropped_tags": int(val_diag["dropped_classes"]),
            })

            if val_pr > best_pr + 1e-4:
                best_pr = float(val_pr)
                bad_epochs = 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad_epochs += 1
                if bad_epochs >= CFG["early_stop_patience"]:
                    print(f"Early stopping at epoch {epoch+1} (best val PR-AUC={best_pr:.4f})")
                    break

            print(
                f"epoch {epoch+1:02d} | time={dt:.1f}s | train_loss={train_loss:.4f} | "
                f"val_ROC-AUC={val_roc:.4f} | val_PR-AUC={val_pr:.4f} | "
                f"valid_tags={val_diag['valid_classes']}/{val_diag['total_classes']}"
            )

        if best_state is not None:
            model.load_state_dict(best_state)

    finally:
        monitor.stop()
        global emissions_kg, resources
        emissions_kg = tracker.stop()
        resources = monitor.summary()

    return history

history = train_loop()
test_roc, test_pr, test_diag = eval_auc(test_loader)

print(f"TEST ROC-AUC (macro): {test_roc:.4f}")
print(f"TEST PR-AUC  (macro): {test_pr:.4f}")
print(f"TEST valid_tags: {test_diag['valid_classes']}/{test_diag['total_classes']} (dropped={test_diag['dropped_classes']})")
print("Emissions (kg CO2eq):", emissions_kg)
print("Resources summary:", resources)

[codecarbon INFO @ 17:33:13] offline tracker init
[codecarbon WARNING @ 17:33:13] Multiple instances of codecarbon are allowed to run at the same time.


Epoch 1/3:   0%|          | 0/250 [00:00<?, ?it/s]

epoch 01 | time=100.7s | train_loss=0.1262 | val_ROC-AUC=0.5337 | val_PR-AUC=0.0359 | valid_tags=186/188


Epoch 2/3:   0%|          | 0/250 [00:00<?, ?it/s]

epoch 02 | time=99.6s | train_loss=0.0739 | val_ROC-AUC=0.6153 | val_PR-AUC=0.0397 | valid_tags=186/188


Epoch 3/3:   0%|          | 0/250 [00:00<?, ?it/s]

epoch 03 | time=99.6s | train_loss=0.0731 | val_ROC-AUC=0.6323 | val_PR-AUC=0.0436 | valid_tags=186/188
TEST ROC-AUC (macro): 0.6392
TEST PR-AUC  (macro): 0.0417
TEST valid_tags: 188/188 (dropped=0)
Emissions (kg CO2eq): 0.0001600073648747084
Resources summary: {'wall_time_s': 299.1053659915924, 'cpu_mean_pct': 42.94782608695652, 'cpu_max_pct': 91.3, 'ram_mean_mb': 707.8416596989966, 'ram_max_mb': 727.640625, 'n_samples': 299, 'accel_mem_mean_mb': 12.900375111047241, 'accel_mem_max_mb': 122.214599609375}


# <h2 style="text-align: center; font-weight: bold;">FINAL TEST</h2>


In [17]:
print(f"TEST ROC-AUC (macro): {test_roc:.4f}")
print(f"TEST PR-AUC  (macro): {test_pr:.4f}")

TEST ROC-AUC (macro): 0.6392
TEST PR-AUC  (macro): 0.0417


# <h3 style="text-align: left; font-weight: bold;">CODECARBON PLUS TRACKED TRAINING</h3>


In [18]:
print("Emissions (kg CO2eq):", emissions_kg)
print("Resources summary:", resources)
print(f"TEST ROC-AUC (macro): {test_roc:.4f}")
print(f"TEST PR-AUC  (macro): {test_pr:.4f}")

Emissions (kg CO2eq): 0.0001600073648747084
Resources summary: {'wall_time_s': 299.1053659915924, 'cpu_mean_pct': 42.94782608695652, 'cpu_max_pct': 91.3, 'ram_mean_mb': 707.8416596989966, 'ram_max_mb': 727.640625, 'n_samples': 299, 'accel_mem_mean_mb': 12.900375111047241, 'accel_mem_max_mb': 122.214599609375}
TEST ROC-AUC (macro): 0.6392
TEST PR-AUC  (macro): 0.0417


# <h2 style="text-align: center; font-weight: bold;">SUMMARY</h2>


In [19]:
from pandas import DataFrame
DataFrame(history).tail()

,epoch,train_loss,val_roc,val_pr,epoch_time_s,val_valid_tags,val_total_tags,val_dropped_tags
0,1,0.126222,0.533670,0.035887,100.703265,186,188,2
1,2,0.073926,0.615343,0.039664,99.565372,186,188,2
2,3,0.073060,0.632264,0.043570,99.556630,186,188,2


In [20]:
print(color.BOLD + "\nFINAL SUMMARY" + color.END)
print(f"Device: {device}")
print(f"Train items: {len(train_df)} | Val items: {len(val_df)} | Test items: {len(test_df)}")
print(f"Tags/classes: {n_classes}")
print(f"TEST ROC-AUC (macro): {test_roc:.4f}")
print(f"TEST PR-AUC  (macro): {test_pr:.4f}")
print(f"Emissions (kg CO2eq): {emissions_kg}")
for k,v in resources.items():
    print(f"{k}: {v}")


FINAL SUMMARY
Device: mps
Train items: 8000 | Val items: 2000 | Test items: 2000
Tags/classes: 188
TEST ROC-AUC (macro): 0.6392
TEST PR-AUC  (macro): 0.0417
Emissions (kg CO2eq): 0.0001600073648747084
wall_time_s: 299.1053659915924
cpu_mean_pct: 42.94782608695652
cpu_max_pct: 91.3
ram_mean_mb: 707.8416596989966
ram_max_mb: 727.640625
n_samples: 299
accel_mem_mean_mb: 12.900375111047241
accel_mem_max_mb: 122.214599609375


# <h2 style="text-align: center; font-weight: bold;">REPORT</h2>
